# YouTube Playlist Tracker Notebook
Run the project workflow interactively from this notebook.

## What this notebook does
- Discover public playlists for a channel
- Fetch per-video stats for one playlist
- Optionally save a snapshot to local history
- Show historical snapshot summaries

Run cells from top to bottom.

In [ ]:
# Optional install (uncomment if needed)
%pip install -q -r requirements.txt

import sys
from pathlib import Path

import pandas as pd

from youtube_tracker.storage import append_snapshot, load_snapshot_history
from youtube_tracker.youtube_api import (
    YouTubeApiError,
    discover_channel_playlists,
    extract_playlist_id,
    fetch_playlist_videos_with_stats,
)

print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)

In [ ]:
# Configuration
API_KEY = ""  # Paste key here or load from env/secrets
CHANNEL_INPUT = ""  # Example: https://www.youtube.com/@TEDx or UC...
PLAYLIST_INPUT = ""  # Example: https://www.youtube.com/playlist?list=PL...
SAVE_SNAPSHOT = True

if not API_KEY:
    raise ValueError('Set API_KEY in this cell before running discovery/fetch.')

repo_root = Path.cwd()
print('Repo root:', repo_root)

In [ ]:
# Discover public playlists for a channel
playlists_df = pd.DataFrame()

if CHANNEL_INPUT.strip():
    try:
        playlists = discover_channel_playlists(API_KEY, CHANNEL_INPUT)
        playlists_df = pd.DataFrame(playlists)
        print(f'Discovered {len(playlists_df)} playlists')
        display(playlists_df.head(20))
    except YouTubeApiError as exc:
        raise RuntimeError(f'Playlist discovery failed: {exc}') from exc
else:
    print('CHANNEL_INPUT is blank; skipping discovery.')

In [ ]:
# Choose playlist to fetch
selected_playlist_id = ""

if PLAYLIST_INPUT.strip():
    selected_playlist_id = extract_playlist_id(PLAYLIST_INPUT.strip())
    if not selected_playlist_id:
        raise ValueError('PLAYLIST_INPUT is not a valid playlist URL or ID.')
elif not playlists_df.empty:
    selected_playlist_id = playlists_df.iloc[0]['playlist_id']
    print('PLAYLIST_INPUT not set. Using first discovered playlist:', selected_playlist_id)
else:
    raise ValueError('Set PLAYLIST_INPUT or provide CHANNEL_INPUT that returns playlists.')

selected_playlist_id

In [ ]:
# Fetch video statistics for selected playlist
try:
    stats_df = fetch_playlist_videos_with_stats(API_KEY, selected_playlist_id)
except YouTubeApiError as exc:
    raise RuntimeError(f'Video stats fetch failed: {exc}') from exc

print(f'Rows: {len(stats_df)}')
display(stats_df.head(20))

In [ ]:
# Optionally save snapshot to local history
snapshot_time = None
if SAVE_SNAPSHOT and not stats_df.empty:
    playlist_title = ""
    if not playlists_df.empty and selected_playlist_id in set(playlists_df['playlist_id']):
        playlist_title = playlists_df.loc[playlists_df['playlist_id'] == selected_playlist_id, 'title'].iloc[0]
    snapshot_time = append_snapshot(
        playlist_id=selected_playlist_id,
        playlist_title=playlist_title,
        df=stats_df,
    )
    print('Snapshot saved at:', snapshot_time)
else:
    print('Snapshot not saved (SAVE_SNAPSHOT is False or no rows).')

In [ ]:
# Historical summary for selected playlist
history_df = load_snapshot_history()

if history_df.empty:
    print('No snapshot history found yet.')
else:
    selected_history = history_df[history_df['playlist_id'] == selected_playlist_id].copy()
    if selected_history.empty:
        print('No snapshots found for selected playlist yet.')
    else:
        summary_df = (
            selected_history.groupby('snapshot_time', as_index=False)
            .agg(
                video_rows=('video_id', 'count'),
                total_views=('view_count', 'sum'),
                total_likes=('like_count', 'sum'),
                total_comments=('comment_count', 'sum'),
            )
            .sort_values('snapshot_time', ascending=False)
        )
        display(summary_df.head(30))